In [ ]:
%load_ext autoreload
%autoreload 2

# core.dom.summary


>DOM summary 
output-file: core.dom.summary
title: core.dom.summary

This notebook provides utility functions for DOM summary
---

In [ ]:
# | default_exp core.dom.summary

In [ ]:
# | export
from IPython.core.interactiveshell import InteractiveShell
from nbdev.showdoc import *

shell = InteractiveShell.instance()
shell.ast_node_interactivity = "all"

In [ ]:
#| export
from fastcore.test import *


In [ ]:
# | export
import json
import os
import re
from collections.abc import Callable
from pathlib import Path
from typing import Optional


from ollama import AsyncClient
try:
    from openai import AsyncOpenAI
except ModuleNotFoundError:
    class AsyncOpenAI:  # type: ignore[no-redef]
        pass

import jsoncfg
from jsoncfg.config_classes import (
    ConfigJSONArray,
    ConfigJSONObject,
    ConfigJSONScalar,
    ConfigNode,
)

from ribosome.core.dom.utils import (
    get_summary_response_async, 
    get_image_summary_response_async, 
    get_text_summary_response_async,
    image_link_pattern,
    get_image_summary_async,
)

from ribosome.core.dom.utils import _config_error,with_async_context

In [ ]:
#| export
from dotenv import load_dotenv

load_dotenv()
OLLAMA_API_KEY= os.getenv('OLLAMA_API_KEY')
DASHSCOPE_API_KEY= os.getenv('DASHSCOPE_API_KEY')

In [ ]:
# | hide
import asyncio
import io
import threading
from contextlib import redirect_stdout
from tempfile import TemporaryDirectory
from unittest.mock import patch

def run_async(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    result = {}
    error = {}

    def _runner():
        try:
            result['value'] = asyncio.run(coro)
        except BaseException as exc:
            error['value'] = exc

    thread = threading.Thread(target=_runner, daemon=True)
    thread.start()
    thread.join()
    if 'value' in error:
        raise error['value']
    return result.get('value')

def make_config(payload):
    return jsoncfg.loads_config(json.dumps(payload, ensure_ascii=False))

In [ ]:
# | export
async def get_leaf_summary_async(node:str, min_len:int, client: AsyncClient | AsyncOpenAI, model:str) -> str: # If the string length is less than 200, return the node as is
    """ Generate a summary for a given text node.

    Args:
        node (str): The text node to summarize.
        min_len (int): The minimum length of the text node to summarize.

    Returns:
        str: The summary of the text node.
    """
    
    if len(node) < min_len:
        return node
    # If the node is not an image link, return a summary of the text using unified function
    response_content = await get_summary_response_async(client, node, model=model, role="user", lang='zh')
    if response_content:
        return response_content
    else:
        # If the response is empty, return the original node
        return node


In [ ]:
# | hide
def test_get_leaf_summary_async_threshold_and_fallback():
    async def fake_summary(client, content, model, role, lang):
        return f'SUM<{content}>'

    with patch.dict(globals(), {'get_summary_response_async': fake_summary}):
        test_eq(run_async(get_leaf_summary_async('hi', 5, object(), 'm')), 'hi')
        test_eq(run_async(get_leaf_summary_async('hello', 3, object(), 'm')), 'SUM<hello>')

    async def empty_summary(client, content, model, role, lang):
        return ''

    with patch.dict(globals(), {'get_summary_response_async': empty_summary}):
        test_eq(run_async(get_leaf_summary_async('hello', 3, object(), 'm')), 'hello')


test_get_leaf_summary_async_threshold_and_fallback()

In [ ]:
# | export
async def get_list_summary_async(root: list, leaf_min_len: int, client: AsyncClient | AsyncOpenAI, model:str) -> str:
    """
    Given a list of strings, return a summary of the list.
    If the list is empty, return an empty string.
    If the list has only one element, return the summary of that element.
    If the list has more than one element, return a summary of the concatenated elements.
    """
    list_summary = []
    for n in root:
        if isinstance(n, str):
            # If the element is a string, get its summary
            summary = await get_leaf_summary_async(n, min_len=leaf_min_len, client=client, model=model)
        elif isinstance(n, int) or isinstance(n, float):
            # If the element is a number, get its summary
            summary = await get_leaf_summary_async(str(n), min_len=leaf_min_len, client=client, model=model)
        elif isinstance(n, dict):
            # If the dict has a 's' key, use it as the summary
            summary = n.get('s', '')
        elif isinstance(n, list):
            # If the element is a list, get its summary
            summary = await get_list_summary_async(n, leaf_min_len=leaf_min_len, client=client, model=model)
        elif n is None:
            # If the element is None, skip it
            summary = ""
        else:
            raise ValueError(f"Unsupported element type: {type(n)} in {n}")
        list_summary.append(summary)
        
    # Concatenate all elements and summarize
    # concatenated = " ".join(list_summary)
    # response = get_text_summary_response(concatenated, model="gemma3:27b", role="user", lang='zh')
    # if response.message.content:
    #     return response.message.content
    # else:
    #     # If the response is empty, raise an exception
    #     raise ValueError("Summary response is empty. Please check the input data.")
    return await get_leaf_summary_async(" ".join(list_summary), min_len=leaf_min_len, client=client, model=model)


In [ ]:
# | hide
def test_get_list_summary_async_handles_supported_values():
    async def identity_leaf(node, min_len, client, model):
        return node

    with patch.dict(globals(), {'get_leaf_summary_async': identity_leaf}):
        result = run_async(get_list_summary_async(['a', 2, {'s': 'dict'}, ['nested'], None], 0, object(), 'm'))

    test_eq(result.strip(), 'a 2 dict nested')
    test_fail(lambda: run_async(get_list_summary_async([('bad',)], 0, object(), 'm')), contains='Unsupported element type')


test_get_list_summary_async_handles_supported_values()

In [ ]:
# | export
def _resolve_image_link(node: dict, config_node: ConfigNode, root_path: Path) -> str:
    """Resolve the link of an image node."""
    try:
        image_link = root_path / node['c'][2][0]
    except (IndexError, KeyError, TypeError) as exc:
        raise _config_error('Invalid image node structure', config_node, node) from exc

    image_link = str(image_link)
    if not image_link or not re.search(image_link_pattern, image_link):
        raise _config_error(f'Invalid image link: {image_link}', config_node, node)
    return image_link


In [ ]:
# | hide
def test_resolve_image_link_validates_shape_and_extension():
    root = Path('/tmp/root')
    test_eq(
        _resolve_image_link({'c': [None, None, ['robot.png', '']]}, None, root),
        str(root / 'robot.png'),
    )
    test_fail(
        lambda: _resolve_image_link({'c': []}, None, root),
        contains='Invalid image node structure',
    )
    test_fail(
        lambda: _resolve_image_link({'c': [None, None, ['robot.txt', '']]}, None, root),
        contains='Invalid image link',
    )


test_resolve_image_link_validates_shape_and_extension()

In [ ]:
# | export
def _config_node_line(config_node: ConfigNode) -> int | str:
    """Get the line number of a ConfigNode."""
    
    try:
        return jsoncfg.node_location(config_node).line
    except Exception:
        return "unknown"

# def _config_error(message: str, config_node: ConfigNode, node=None) -> ValueError:
#     """Create a ValueError with detailed information about a ConfigNode."""
#     details = [
#         message,
#         f"ConfigNode: {config_node} of type {type(config_node)}.",
#         f"On line: {_config_node_line(config_node)}",
#     ]
#     if node is not None:
#         details.insert(1, f"Node: {node}")
#     return ValueError("\n".join(details))


In [ ]:
# | hide
def test_config_node_line_returns_line_or_unknown():
    config = make_config({'node': {'value': 1}})['node']['value']
    test_eq(isinstance(_config_node_line(config), int), True)
    test_eq(_config_node_line(None), 'unknown')


test_config_node_line_returns_line_or_unknown()

In [ ]:
# | export
@with_async_context(lambda config_node, node: str(_config_error("Error summarizing image node", config_node, node)))
async def _summarize_image_node_async(config_node: ConfigNode,node: dict,root_path: Path, client: AsyncClient|AsyncOpenAI, model: str) -> dict:
    """Summarize an image node asynchronously.

    Args:
        self (DOMClass): The DOMClass instance.
        node (dict): The image node.
        config_node (ConfigNode): The configuration node associated with the image.

    Returns:
        dict: The summarized image node.
    """
    if not isinstance(node.get('c', []), list):
        raise _config_error("Invalid image caption format", config_node, node)

    image_link = _resolve_image_link(node, config_node,root_path)
    node["s"] = await get_image_summary_async(
        client=client,
        image_link=image_link,
        model=model,
        role="user",
        lang='zh'
    )
    print(f"Summarize image: {image_link} on line {_config_node_line(config_node)}")
    return node


In [ ]:
# | hide
def test_summarize_image_node_async_sets_summary_and_validates_caption():
    async def fake_image_summary(client, image_link, model, role, lang):
        return f'IMG<{Path(image_link).name}>'

    with TemporaryDirectory() as tmpdir:
        root = Path(tmpdir)
        node = {'t': 'Image', 'c': [['', [], []], [], ['robot.png', '']]}
        stdout = io.StringIO()
        with patch.dict(globals(), {'get_image_summary_async': fake_image_summary}):
            with redirect_stdout(stdout):
                result = run_async(_summarize_image_node_async(None, node, root, object(), 'm'))

    test_eq(result['s'], 'IMG<robot.png>')
    test_eq('Summarize image:' in stdout.getvalue(), True)
    test_fail(lambda: run_async(_summarize_image_node_async(None, {'t': 'Image', 'c': 'bad'}, Path('.'), object(), 'm')), contains='Invalid image caption format')


test_summarize_image_node_async_sets_summary_and_validates_caption()

In [ ]:
# | export
async def _summarize_leaf_async(node: str, leaf_min_len: int, min_len: Optional[int], client: AsyncClient|AsyncOpenAI, model: str, lang: str) -> str:
    """Summarize a leaf node asynchronously."""
    text = str(node).strip()
    threshold = leaf_min_len if min_len is None else min_len
    if len(text) < threshold:
        return text

    response_content = await get_summary_response_async(
        client,
        text,
        model=model,
        role='user',
        lang=lang,
    )
    return response_content or text


In [ ]:
# | hide
def test_summarize_leaf_async_respects_threshold_and_fallback():
    async def fake_summary(client, text, model, role, lang):
        return f'TXT<{text}>'

    with patch.dict(globals(), {'get_summary_response_async': fake_summary}):
        test_eq(run_async(_summarize_leaf_async('hi', 5, None, object(), 'm', 'zh')), 'hi')
        test_eq(run_async(_summarize_leaf_async('hello', 1, None, object(), 'm', 'zh')), 'TXT<hello>')

    async def empty_summary(client, text, model, role, lang):
        return ''

    with patch.dict(globals(), {'get_summary_response_async': empty_summary}):
        test_eq(run_async(_summarize_leaf_async('hello', 1, None, object(), 'm', 'zh')), 'hello')


test_summarize_leaf_async_respects_threshold_and_fallback()

In [ ]:
# | export
async def _coerce_summary_async(value: str|int|float|dict|list|None, leaf_min_len: int, min_len: Optional[int], client: AsyncClient|AsyncOpenAI, model: str, lang: str) -> str:
    """Coerce a value into a summarized string.

    Args:
        self (DOMClass): The DOMClass instance.
        value: The value to be coerced into a summarized string.

    Raises:
        ValueError: If the value type is unsupported.

    Returns:
        str: The summarized string.
    """
    if isinstance(value, str):
        return await _summarize_leaf_async(value, leaf_min_len=0, min_len=None, client=client, model=model, lang=lang)
    if isinstance(value, (int, float)):
        return await _summarize_leaf_async(str(value), leaf_min_len=0, min_len=None, client=client, model=model, lang=lang)
    if isinstance(value, dict):
        return value.get('s', '')
    if isinstance(value, list):
        return await _summarize_list_async(value, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang)
    if value is None:
        return ""
    raise ValueError(f"Unsupported element type: {type(value)} in {value}")


In [ ]:
# | hide
def test_coerce_summary_async_handles_supported_shapes():
    async def fake_leaf(node, leaf_min_len, min_len, client, model, lang):
        return f'LEAF<{node}>'

    async def fake_list(value, leaf_min_len, min_len, client, model, lang):
        return f'LIST<{len(value)}>'

    with patch.dict(globals(), {'_summarize_leaf_async': fake_leaf, '_summarize_list_async': fake_list}):
        test_eq(run_async(_coerce_summary_async('x', 1, 2, object(), 'm', 'zh')), 'LEAF<x>')
        test_eq(run_async(_coerce_summary_async(2, 1, 2, object(), 'm', 'zh')), 'LEAF<2>')
        test_eq(run_async(_coerce_summary_async({'s': 'dict'}, 1, 2, object(), 'm', 'zh')), 'dict')
        test_eq(run_async(_coerce_summary_async(['a', 'b'], 1, 2, object(), 'm', 'zh')), 'LIST<2>')
        test_eq(run_async(_coerce_summary_async(None, 1, 2, object(), 'm', 'zh')), '')

    test_fail(lambda: run_async(_coerce_summary_async({1, 2}, 1, 2, object(), 'm', 'zh')), contains='Unsupported element type')


In [ ]:
# | export
async def _summarize_list_async(
    root: list,
    leaf_min_len: int,
    min_len: Optional[int],
    client: AsyncClient|AsyncOpenAI,
    model: str,
    lang: str
) -> str:
    """Summarize a list of values asynchronously."""
    parts = [await _coerce_summary_async(item, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang) for item in root]
    merged = ' '.join(part for part in parts if part)
    return await _summarize_leaf_async(merged, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang)


In [ ]:
# | hide
def test_summarize_list_async_merges_nonempty_parts():
    async def fake_coerce(item, leaf_min_len, min_len, client, model, lang):
        return {'a': 'A', 'b': '', 'c': 'C'}[item]

    async def fake_leaf(node, leaf_min_len, min_len, client, model, lang):
        return f'SUM<{node}>'

    with patch.dict(globals(), {'_coerce_summary_async': fake_coerce, '_summarize_leaf_async': fake_leaf}):
        result = run_async(_summarize_list_async(['a', 'b', 'c'], 0, None, object(), 'm', 'zh'))

    test_eq(result, 'SUM<A C>')


test_coerce_summary_async_handles_supported_shapes()
test_summarize_list_async_merges_nonempty_parts()

In [ ]:
# | export
def _record_summary_progress(node_type: str, config_node: ConfigNode, file_path: Path, table_count: int, section_count: int) -> None:
    """Record the progress of summarizing a node.

    Args:
        node_type (str): The type of the node being summarized.
        config_node (ConfigNode): The configuration node associated with the summary.
    """
    if node_type == "Table":
        print(f"{file_path} Summarize table: {table_count} on line {_config_node_line(config_node)}")
        table_count += 1
    elif node_type == "Section":
        print(f"{file_path} Summarize section: {section_count} on line {_config_node_line(config_node)}")
        section_count += 1


In [ ]:
# | hide
def test_record_summary_progress_prints_expected_labels():
    config = make_config({'line': 1})['line']
    stdout = io.StringIO()
    with redirect_stdout(stdout):
        _record_summary_progress('Table', config, Path('doc.md'), 2, 3)
        _record_summary_progress('Section', config, Path('doc.md'), 2, 3)

    output = stdout.getvalue()
    test_eq('Summarize table: 2' in output, True)
    test_eq('Summarize section: 3' in output, True)


test_record_summary_progress_prints_expected_labels()

In [ ]:
# | export
@with_async_context(lambda config_node, node: str(_config_error("Error summarizing node", config_node, node)))
async def summary_node_pair_async(
    config_node: ConfigNode,
    node: dict | list | str | int | float | bool | None,
    root_path: Path,
    client: AsyncClient|AsyncOpenAI,
    model: str,
    file_path: Path,
    table_count: int,
    section_count: int,
    action: Optional[Callable] = None,
    leaf_min_len: int = 0,
    min_len: Optional[int] = None,
    lang: str='zh',
) -> tuple[ConfigNode, dict | list | str | int | float | bool | None]:
    """ Generate a summary for a given node.

    Args:
        config_node (ConfigNode): The configuration node associated with the node to summarize.
        node (dict | list | str | int | float | bool | None): The node to summarize.
        action (Optional[Callable], optional): An optional action to apply to the node before summarizing. Defaults to None.

    Raises:
        self._config_error: Raises an error if the node cannot be summarized.
        
    Returns:
        tuple[ConfigNode, dict | list | str | int | float | bool | None]: A tuple containing the configuration node and the summarized node.
    """

    if action:
        node = action(node)

    if isinstance(node, dict):
        if not isinstance(config_node, ConfigJSONObject):
            raise _config_error(f"Expected ConfigJSONObject, got {type(config_node)}", config_node, node)

        t = node.get("t")
        if t is None:
            raise _config_error("Node does not have a 't' key", config_node, node)

        if t == "Image":
            node = await _summarize_image_node_async(config_node, node, root_path=root_path, client=client,model=model)
        elif t in {"Cite", "AlignDefault", "ColWidth"}:
            node["s"] = ""
        else:
            content = node.get("c")
            if not content:
                node["s"] = ""
            else:
                dict_summary = []
                for (_, cvalue), (key, value) in zip(config_node, node.items()):
                    if key == "t":
                        continue
                    if isinstance(cvalue, ConfigJSONArray):
                        if not isinstance(value, list):
                            raise _config_error(f"Expected a list for key {key}, got {type(value)}", cvalue, value)
                        if not value:
                            continue
                        children = []
                        for citem, item in zip(cvalue, value):
                            _, child = await summary_node_pair_async(citem, item, file_path=file_path, table_count=table_count, section_count=section_count, root_path=root_path, client=client, model=model)
                            children.append(child)
                        node[key] = children
                        dict_summary.append(await _summarize_list_async(children, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang))
                    elif isinstance(cvalue, ConfigJSONObject):
                        if not isinstance(value, dict):
                            raise _config_error(f"Expected a dict for key {key}, got {type(value)}", cvalue, value)
                        _, child = await summary_node_pair_async(cvalue, value, file_path=file_path, table_count=table_count, section_count=section_count, root_path=root_path, client=client, model=model)
                        node[key] = child
                        if isinstance(child, dict) and 's' in child:
                            dict_summary.append(child['s'])
                    elif value is None or value == "":
                        continue
                    elif isinstance(cvalue, ConfigJSONScalar):
                        if not isinstance(value, (str, int, float, bool)):
                            raise _config_error(f"Expected a scalar for key {key}, got {type(value)}", cvalue, value)
                        dict_summary.append(await _summarize_leaf_async(str(value), leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang))

                joined_summary = " ".join(part for part in dict_summary if part)
                node["s"] = await _summarize_leaf_async(joined_summary, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang) if joined_summary else ""

            _record_summary_progress(t, config_node, file_path=file_path, table_count=table_count, section_count=section_count)

    elif isinstance(node, list):
        if not isinstance(config_node, ConfigJSONArray):
            raise _config_error(f"Expected ConfigJSONArray, got {type(config_node)}", config_node, node)
        new_node = []
        for citem, item in zip(config_node, node):
            if isinstance(item, (dict, list)):
                _, child = await summary_node_pair_async(citem, item, file_path=file_path, table_count=table_count, section_count=section_count, root_path=root_path, client=client, model=model)
            elif isinstance(item, (str, int, float, bool)):
                child = str(item).strip()
            else:
                child = item
            new_node.append(child)
        node = new_node

    elif isinstance(node, (str, int, float, bool)):
        if not isinstance(config_node, ConfigJSONScalar):
            raise _config_error(f"Expected ConfigJSONScalar, got {type(config_node)}", config_node, node)
        node = await _summarize_leaf_async(str(node), leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang)
    elif node is not None:
        raise _config_error(f"Unsupported node type: {type(node)}", config_node, node)

    return config_node, node

In [ ]:
# | hide
def test_summary_node_pair_async_summarizes_dict_list_and_scalar_paths():
    payload = {'t': 'Section', 'c': [{'t': 'Header', 'c': [2]}, 'hello', {'t': 'Cite', 'c': []}]}
    config = make_config(payload)
    progress = []

    async def fake_leaf(node, leaf_min_len, min_len, client, model, lang):
        return f'S<{node}>'

    async def fake_list(root, leaf_min_len, min_len, client, model, lang):
        return f'L<{len(root)}>'

    def fake_progress(node_type, config_node, file_path, table_count, section_count):
        progress.append((node_type, table_count, section_count))

    with patch.dict(globals(), {'_summarize_leaf_async': fake_leaf, '_summarize_list_async': fake_list, '_record_summary_progress': fake_progress}):
        _, result = run_async(
            summary_node_pair_async(config, json.loads(json.dumps(payload)), Path('.'), object(), 'm', Path('doc.md'), 1, 2)
        )

    test_eq(result['c'][0]['s'], 'S<L<1>>')
    test_eq(result['c'][1], 'S<hello>')
    test_eq(result['c'][2]['s'], '')
    test_eq(result['s'], 'S<L<3>>')
    test_eq(progress, [('Header', 1, 2), ('Section', 1, 2)])
    test_fail(lambda: run_async(summary_node_pair_async(make_config({}), {}, Path('.'), object(), 'm', Path('doc.md'), 0, 0)), contains="Node does not have a 't' key")


test_summary_node_pair_async_summarizes_dict_list_and_scalar_paths()

In [ ]:
# | export
async def summary_node_async(
        node: dict | list,
        root_path: Path,
        file_path: Path,
        client: AsyncClient|AsyncOpenAI,
        model: str,
        table_count: int,
        section_count: int,
        action: Optional[Callable] = None,
        leaf_min_len: int = 0,
        min_len: Optional[int] = None,
        lang: str='zh',
    ) -> dict | list:
    '''
    Given a string node, add key,value pair: node['s'] = node_summary, and return the node 
    '''

    if isinstance(node, dict):
        try:
            t = node["t"]
        except KeyError:
            raise ValueError(f"Node does not have a 't' key: {node}")
        if t == "Image":  # Image summary, the Image node is as defined in the pandoc AST
            summary = []
            if (not node['c'][1] == []) and (node['c'][1][0] is not None) and (node['c'][1][0].get('c') is not None):
                summary.append(node['c'][1][0]['c'])  # The content of the second element is the image caption
            try:
                # summary.append(node['c'][0])  # The first element are defined to be attributes of the image rendering, i.e. content-irrelevant.
                # If the node is an image, get its link
                image_link = root_path / node['c'][2][0]  # Assuming the image link is in the third element of the list
                image_link = str(image_link)
                if image_link and re.search(image_link_pattern, image_link):  # re.match leads to empty match if there's spaces in the path!
                    # If the node is an image link, get its summary
                    assert isinstance(client,AsyncClient), "Only ollama client is allowed for image summary"
                    response = await get_image_summary_response_async(
                        client=client,
                        image_link=image_link,
                        model=model,
                        role="user",
                        lang=lang
                        )
                    summary.append(response.message.content)
                else:
                    # If the node is not an image link, summarize its content
                    raise ValueError(f"Invalid image link: {image_link}")
                # Get the summary of the image caption
            except (IndexError, KeyError):
                # Handle cases where the image link is not in the expected format
                raise ValueError(f"Invalid image node structure: {node}")
            # The second element is the image title
            summary.append(node['c'][2][1])  # The second element is the image title
            assert isinstance(client,AsyncClient), "Only ollama client is allowed for image summary"
            response_txt = await get_text_summary_response_async(
                client=client,
                content=" ".join(summary),
                model=model,
                role="user",
                lang=lang
            )
            node["s"] = response_txt.message.content
            print(f"Summarize image: {image_link}")
        elif t == "Cite" or t == "AlignDefault" or t == "ColWidth" :  # Quoted node will be ignored
            node["s"] = ""  # Set the summary to an empty string
        else: # TextBlock summary
            c = node.get("c", None)
            if not c:
                node["s"] = ""
            else:
                dict_summary = []
                for key, value in node.items(): # get summary of the values (content)
                    if key == "t":
                        continue
                    if isinstance(value, list):  # get the summary of the string list
                        if value == []:
                            # If the list is empty, skip it
                            continue
                        # If the value is a list, summarize each element
                        node[key] = [
                            await summary_node_async(
                                node=child,
                                root_path=root_path,
                                file_path=file_path,
                                client=client,
                                model=model,
                                table_count=table_count,
                                section_count=section_count,
                                action=action,
                                leaf_min_len=leaf_min_len,
                                min_len=min_len,
                                lang=lang
                                )
                            if isinstance(child, (dict,list))
                            else child
                            for child in value
                        ]
                        dict_summary.append(
                            await _summarize_list_async(
                                root=value,
                                leaf_min_len=leaf_min_len,
                                min_len=min_len,
                                client=client,
                                model=model,
                                lang=lang)
                            )
                    elif isinstance(value, dict):
                        child = await summary_node_async(
                            node=value,
                            root_path=root_path,
                            file_path=file_path,
                            client=client,
                            model=model,
                            table_count=table_count,
                            section_count=section_count,
                            action=action,
                            leaf_min_len=leaf_min_len,
                            min_len=min_len,
                            lang=lang
                        )  # insert the value['s']
                        assert isinstance(child, dict) and 's' in child, f"Expected dict with 's' key, got {child}"
                        dict_summary.append(child['s'])
                    elif value is None or value == "":
                        # If the value is None, skip it
                        continue
                    else:
                        if not isinstance(value, str):
                            # If the value is not a string, convert it to a string
                            value = str(value)
                        dict_summary.append(
                            await _summarize_leaf_async(
                                node=value,
                                leaf_min_len=leaf_min_len,
                                min_len=min_len,
                                client=client,
                                model=model,
                                lang=lang
                                )
                            )
                # get the summary of the node
                if dict_summary:  # type: ignore
                    # If there are summaries, concatenate them
                    node["s"] = await _summarize_leaf_async(
                        node=" ".join(dict_summary),
                        leaf_min_len=leaf_min_len,
                        min_len=min_len,
                        client=client,
                        model=model,
                        lang=lang
                    )
                    # node["s"] = get_text_summary_response(
                    #     " ".join(dict_summary), model="gemma3:27b", role="user", lang='zh'
                    # ).message.content
                else:
                    # If no summaries, set to empty string
                    node["s"] = ""
            # if t is table
            if t == "Table":
                print(f"{file_path} Summarize table: {table_count}")
                table_count += 1
            elif t == "Section":
                print(f"{file_path} Summarize section: {section_count} section depth {node['c'][0]['c'][0]}")
                section_count += 1
    elif isinstance(node, list):
        node = [
            await summary_node_async(
                node=child,
                root_path=root_path,
                file_path=file_path,
                client=client,
                model=model,
                table_count=table_count,
                section_count=section_count,
                action=action,
                leaf_min_len=leaf_min_len,
                min_len=min_len,
                lang=lang
                ) if isinstance(child, (dict,list)) else child
            for child in node
        ]
    else:
        assert isinstance(node, str), f"Expected a string node, got {type(node)}, on json line"
        # If the node is a string, get its summary
        node_summary = await _summarize_leaf_async(
            node=node,
            leaf_min_len=leaf_min_len,
            min_len=min_len,
            client=client,
            model=model,
            lang=lang
        )
        node = node_summary
    return node

In [ ]:
# | hide
def test_summary_node_async_summarizes_legacy_node_shapes():
    payload = {'t': 'Section', 'c': [{'t': 'Header', 'c': [2]}, 'hello', {'t': 'Cite', 'c': []}]}

    async def fake_leaf(node, leaf_min_len, min_len, client, model, lang):
        return f'S<{node}>'

    async def fake_list(root, leaf_min_len, min_len, client, model, lang):
        return 'LIST'

    with patch.dict(globals(), {'_summarize_leaf_async': fake_leaf, '_summarize_list_async': fake_list}):
        result = run_async(summary_node_async(json.loads(json.dumps(payload)), Path('.'), Path('doc.md'), object(), 'm', 1, 2))

    test_eq(result['s'], 'S<LIST>')
    test_eq(result['c'][0]['s'], 'S<LIST>')
    test_eq(result['c'][1], 'hello')
    test_eq(result['c'][2]['s'], '')
    test_fail(lambda: run_async(summary_node_async({}, Path('.'), Path('doc.md'), object(), 'm', 0, 0)), contains="Node does not have a 't' key")


test_summary_node_async_summarizes_legacy_node_shapes()

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()